In [1]:
import os
import glob
from typing import Dict, List, Optional, Tuple

import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.crs import CRS


# ----------------------------
# User parameters
# ----------------------------
SAFE_DIR = "/home/stagiaire/Téléchargements/S2B_MSIL2A_20210508T135109_N0500_R024_T21HUB_20230304T174146.SAFE"

# If None, keep the native CRS of the reference 10 m band (recommended).
# Example: CRS.from_epsg(32723)
TARGET_CRS: Optional[CRS] = None

# Output resolution in meters (Sentinel-2 10 m target grid)
TARGET_RES: float = 10.0

# Output GeoTIFF path (single multiband stack)
OUT_TIF = "./s2_l2a_stack_12bands_10m_float32_buenos_aires_test.tif"

# Sentinel-2 L2A scale factor (DN = reflectance * 10000)
S2_SCALE = 10000.0

# Band list (12 bands)
BANDS = ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B09", "B11", "B12"]


# ----------------------------
# Helpers
# ----------------------------
def find_band_jp2_paths(safe_dir: str, bands: List[str]) -> Dict[str, str]:
    """
    Find JP2 file paths for requested Sentinel-2 bands inside a .SAFE directory.

    Parameters
    ----------
    safe_dir : str
        Path to the .SAFE directory.
    bands : list[str]
        Band identifiers (e.g., "B02", "B8A").

    Returns
    -------
    dict[str, str]
        Mapping {band: jp2_path}.

    Raises
    ------
    FileNotFoundError
        If any band cannot be found.
    """
    paths: Dict[str, str] = {}
    for b in bands:
        pattern = os.path.join(safe_dir, "**", f"*_{b}_*.jp2")
        matches = sorted(glob.glob(pattern, recursive=True))

        # Prefer IMG_DATA paths and prefer the native resolution folder when multiple matches exist.
        # For example, B02 often exists as *_B02_10m.jp2 in IMG_DATA/R10m.
        if len(matches) == 0:
            raise FileNotFoundError(f"Could not find JP2 for band {b} using pattern: {pattern}")

        def score(p: str) -> Tuple[int, int]:
            p_u = p.upper()
            img_data_score = 1 if "IMG_DATA" in p_u else 0
            # Prefer native folders: R10m for 10m bands, R20m for 20m bands, R60m for 60m bands
            # This is just a preference heuristic.
            native_score = 0
            if b in {"B02", "B03", "B04", "B08"} and "R10M" in p_u:
                native_score = 3
            elif b in {"B05", "B06", "B07", "B8A", "B11", "B12"} and "R20M" in p_u:
                native_score = 3
            elif b in {"B01", "B09"} and "R60M" in p_u:
                native_score = 3
            return (img_data_score + native_score, len(p))

        best = sorted(matches, key=score, reverse=True)[0]
        paths[b] = best

    return paths


def build_target_grid(
    ref_path: str,
    target_crs: Optional[CRS],
    target_res: float,
) -> Tuple[CRS, rasterio.Affine, int, int]:
    """
    Build a target grid (CRS, transform, width, height) at the requested resolution.

    The grid is derived from the reference raster bounds, optionally reprojected to target_crs.

    Parameters
    ----------
    ref_path : str
        Path to a reference raster (typically a 10 m band like B02).
    target_crs : CRS or None
        Target CRS. If None, uses the reference raster CRS.
    target_res : float
        Target pixel size in meters.

    Returns
    -------
    (CRS, Affine, int, int)
        target_crs, target_transform, target_width, target_height
    """
    with rasterio.open(ref_path) as src:
        src_crs = src.crs
        dst_crs = src_crs if target_crs is None else target_crs

        if dst_crs == src_crs:
            # Keep CRS; snap to requested resolution using the reference bounds
            left, bottom, right, top = src.bounds
            dst_transform = rasterio.Affine(
                target_res, 0.0, left,
                0.0, -target_res, top
            )
            dst_width = int(np.ceil((right - left) / target_res))
            dst_height = int(np.ceil((top - bottom) / target_res))
            return dst_crs, dst_transform, dst_width, dst_height

        # Reproject bounds/grid to target CRS
        dst_transform, dst_width, dst_height = calculate_default_transform(
            src_crs, dst_crs, src.width, src.height, *src.bounds, resolution=target_res
        )
        return dst_crs, dst_transform, dst_width, dst_height


def reproject_resample_to_grid(
    src_path: str,
    dst_crs: CRS,
    dst_transform: rasterio.Affine,
    dst_width: int,
    dst_height: int,
    resampling: Resampling,
) -> np.ndarray:
    """
    Reproject and resample a single-band raster to a target grid.

    Output is float32.

    Parameters
    ----------
    src_path : str
        Input raster path.
    dst_crs : CRS
        Target CRS.
    dst_transform : Affine
        Target affine transform.
    dst_width : int
        Target width in pixels.
    dst_height : int
        Target height in pixels.
    resampling : Resampling
        Resampling method.

    Returns
    -------
    np.ndarray
        Array of shape (dst_height, dst_width), dtype float32.
    """
    with rasterio.open(src_path) as src:
        src_arr = src.read(1)

        # Many S2 exports use 0 as no-data. If nodata is defined, respect it.
        src_nodata = src.nodata
        if src_nodata is None:
            src_nodata = 0

        dst_arr = np.empty((dst_height, dst_width), dtype=np.float32)

        reproject(
            source=src_arr,
            destination=dst_arr,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src_nodata,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            dst_nodata=np.nan,
            resampling=resampling,
        )

        return dst_arr


def band_resampling_method(band: str) -> Resampling:
    """
    Choose resampling method per band type.

    For radiometric bands, bilinear is typically appropriate.
    """
    return Resampling.bilinear


# ----------------------------
# Build a 12-band 10 m stack (float32 reflectance)
# ----------------------------
band_paths = find_band_jp2_paths(SAFE_DIR, BANDS)

# Use B02 (10 m) as reference grid if available; otherwise fallback to B03.
ref_band = "B02" if "B02" in band_paths else "B03"
ref_path = band_paths[ref_band]

dst_crs, dst_transform, dst_width, dst_height = build_target_grid(
    ref_path=ref_path,
    target_crs=TARGET_CRS,
    target_res=TARGET_RES,
)

stack = np.empty((len(BANDS), dst_height, dst_width), dtype=np.float32)

for i, b in enumerate(BANDS):
    arr = reproject_resample_to_grid(
        src_path=band_paths[b],
        dst_crs=dst_crs,
        dst_transform=dst_transform,
        dst_width=dst_width,
        dst_height=dst_height,
        resampling=band_resampling_method(b),
    )

    # Convert DN to surface reflectance
    arr = arr / S2_SCALE

    stack[i] = arr

# Write multiband GeoTIFF
os.makedirs(os.path.dirname(OUT_TIF), exist_ok=True)

profile = {
    "driver": "GTiff",
    "height": dst_height,
    "width": dst_width,
    "count": len(BANDS),
    "dtype": "float32",
    "crs": dst_crs,
    "transform": dst_transform,
    "nodata": np.nan,
    "tiled": True,
    "compress": "deflate",
    "predictor": 3,
    "interleave": "band",
}

with rasterio.open(OUT_TIF, "w", **profile) as dst:
    for idx, b in enumerate(BANDS, start=1):
        dst.write(stack[idx - 1], idx)
        dst.set_band_description(idx, b)

OUT_TIF


'./s2_l2a_stack_12bands_10m_float32_buenos_aires_test.tif'